In [ ]:

import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point
import bioelements as be


# Random generator
RNG = np.random.default_rng(seed=0)

N_PROCESSES = 10

In [ ]:
OUT = Path("../output")
DATA_DIR = Path("../data_cyano")
N_PROCESSES = 10
OUT.mkdir(exist_ok=True)

In [ ]:
NEW_OUT = OUT / "Crystal_analysis"
NEW_OUT.mkdir(parents=True, exist_ok=True)

PROTEIN_KEYS = ["1JB0-PSI-syn-cocc", "3WU2-PSII-ThermosynVul"]
LATTICE_TYPES = [ "hexagonal","square"]
MAX_VARIATIONS = ["1-2", "1-4", "1-8", "2-0"]


exp_config = be.brownian_lattice.ExperimentLatticeConfig()
exp_config.replicates = 3000
exp_config.diff_coefficient = 3.5e9  # A2/s == 3.5*10^-7 cm2/s
exp_config.nsteps = 1_000_000
exp_config.particle_radius = 5  # Angström # average linear length of PQ is 7A
exp_config.dimensions = (0.0, 5000.0)
exp_config.random_start = True
exp_config.start_area = Point(2500,2500).buffer(300)
exp_config.aim_area = Point(0,0).buffer(500)
exp_config.has_ghost = False
exp_config.shift_origin = True
exp_config.store_history = False
exp_config.save_every = 1000
exp_config.workers = 10

master_csv = NEW_OUT / f"crystal_survival_all_data_ps-{exp_config.particle_radius}.csv"

master_csv.unlink(missing_ok=True)


def process_single_ensemble(file_list, config, key, ltp, mv):
    ens_exp = be.brownian_lattice.EnsembleExperimentLattice(file_list, config)
    run = ens_exp.run()
    df = run.mean_over_runs()
    mean_coverage = run.get_mean_coverage()
    df_summary = df.assign(key=key, ltp=ltp, mv=mv, mean_coverage=mean_coverage)

    try:
        del ens_exp
        del run
    except NameError:
        pass
    gc.collect()

    return df_summary, mean_coverage


for pkey in PROTEIN_KEYS:
    for ltp in LATTICE_TYPES:
        fig, ax = plt.subplots()
        for mv in MAX_VARIATIONS:
            file_lst = list(
                (OUT / (pkey + "_crystal")).glob(f"*_{ltp}_regularTrue_{mv}*.wkt")
            )
            if len(file_lst) == 0:
                print(f"[skip] no files with {pkey} {ltp} {mv}")
                continue

            print(f"Processing {pkey} {ltp} {mv} (files: {len(file_lst)})")

            df, mean_coverage = process_single_ensemble(
                file_list=file_lst, config=exp_config, key=pkey, ltp=ltp, mv=mv
            )

            if "Aim" in df.columns:
                df["Time_ms"] = df["Time"] * 1000
                df.plot(x="Time_ms", y="Aim", label=f"p = {mv.replace('-', '.')}", ax=ax)
                ax.fill_between(
                df["Time_ms"],
                df["Aim_ci_low"],
                df["Aim_ci_high"],
                alpha=0.3
            )

            write_header = not master_csv.exists()
            df.to_csv(master_csv, mode="a", header=write_header, index=False)

            del df
            gc.collect()

        ax.set_title(pkey + " " + ltp)
        ax.set_xlabel("Time/[ms]", size=15)
        ax.set_ylabel("% in Aim", size=15)
        ax.tick_params(labelsize=15)

        fig.savefig(
            NEW_OUT / f"crystal_survival_{pkey}_{ltp}_ps-{exp_config.particle_radius}.png",
            bbox_inches="tight",
        )
        plt.close()
        gc.collect()
